# Steam Games Dataset - Exploratory Data Analysis

## Project Overview

In this notebook, we explore the **Steam Games Dataset (2025)** to understand what the gaming landscape looks like on the world's largest PC gaming platform.

**Goals for this notebook:**
- Get familiar with the dataset — how big is it, what columns do we have, what's missing?
- Look at how game releases have trended over time
- Understand the pricing landscape (free vs paid, price distributions)
- See which genres dominate the platform
- Explore Metacritic scores and community recommendations

This is the first notebook in the series — we're just getting our bearings before diving into deeper analysis.

## 1. Setup & Imports

In [ ]:
# Standard library imports
import sys
import os

# Add the project root to our path so we can import our custom modules
sys.path.insert(0, os.path.abspath(".."))

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# Our custom project modules
from src.data_loader import (
    load_applications,
    build_games_with_genres,
    STEAM_COLORS,
    CHART_COLORS,
)
from src.utils import get_plotly_layout, setup_matplotlib_style

# Apply our Steam-themed matplotlib style
setup_matplotlib_style()

# Make pandas show more columns so we don't miss anything
pd.set_option("display.max_columns", 35)

print("All imports loaded successfully!")

## 2. Load the Data

In [ ]:
# Load the applications dataset (already filtered to games only)
games = load_applications()

# Quick sanity check — let's see the first few rows
print(f"\nDataset loaded with {len(games)} games")
games.head()

## 3. Basic EDA — Understanding the Dataset

In [ ]:
# Let's start with the basics: how big is our dataset and what do we have?
print("=" * 60)
print("DATASET SHAPE")
print("=" * 60)
print(f"Rows (games): {games.shape[0]:,}")
print(f"Columns: {games.shape[1]}")
print()

# List all columns so we know what we're working with
print("=" * 60)
print("COLUMNS & DATA TYPES")
print("=" * 60)
for col in games.columns:
    print(f"  {col:<30} {str(games[col].dtype):<15} | {games[col].nunique()} unique values")

In [ ]:
# Check summary statistics for numeric columns
# This tells us about ranges, averages, and potential outliers
print("=" * 60)
print("SUMMARY STATISTICS (Numeric Columns)")
print("=" * 60)
games.describe()

In [ ]:
# Missing values analysis — what data is incomplete?
# This is important because missing data can skew our analysis
missing_counts = games.isnull().sum()
missing_percent = (games.isnull().sum() / len(games)) * 100

# Combine into a nice summary table
missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing %": missing_percent.round(2),
}).sort_values("Missing %", ascending=False)

# Only show columns that actually have missing data
missing_summary_filtered = missing_summary[missing_summary["Missing Count"] > 0]

print("=" * 60)
print("MISSING VALUES (columns with gaps)")
print("=" * 60)
print(missing_summary_filtered)
print(f"\n{len(missing_summary_filtered)} out of {len(games.columns)} columns have missing data")

In [ ]:
# Visualize missing data with a heatmap
# Yellow bars = missing values, so we can quickly spot problem areas
fig, ax = plt.subplots(figsize=(14, 6))

sns.heatmap(
    games.isnull().T,  # Transpose so columns are on the y-axis
    cbar=True,
    cmap=[STEAM_COLORS["medium_blue"], STEAM_COLORS["accent_orange"]],
    yticklabels=True,
    ax=ax,
)

ax.set_title("Missing Values Heatmap", fontsize=16, pad=15)
ax.set_xlabel("Row Index (sampled)")
ax.set_ylabel("")

plt.tight_layout()
plt.show()

## 4. Distribution Charts

### 4.1 Games Released Per Year
Has Steam been growing? Let's see how many games come out each year.

In [ ]:
# Count games per release year
# Drop NaN years (games with missing release dates)
games_per_year = (
    games["release_year"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
)

# Filter to reasonable range — some entries have weird dates
games_per_year = games_per_year[
    (games_per_year.index >= 2000) & (games_per_year.index <= 2025)
]

print(f"Year range: {games_per_year.index.min()} to {games_per_year.index.max()}")
print(f"Peak year: {games_per_year.idxmax()} with {games_per_year.max():,} games")

# Build the bar chart
fig = go.Figure(
    go.Bar(
        x=games_per_year.index.astype(str),
        y=games_per_year.values,
        marker_color=STEAM_COLORS["light_blue"],
        hovertemplate="Year: %{x}<br>Games: %{y:,}<extra></extra>",
    )
)

fig.update_layout(
    get_plotly_layout(
        "Games Released Per Year on Steam",
        xaxis_title="Year",
        yaxis_title="Number of Games",
    )
)

fig.show()

### 4.2 Price Distribution
How much do paid games typically cost on Steam?

In [ ]:
# Filter to only paid games (exclude free-to-play)
paid_games = games[games["price_dollars"] > 0].copy()

print(f"Total games: {len(games):,}")
print(f"Paid games: {len(paid_games):,}")
print(f"Average price: ${paid_games['price_dollars'].mean():.2f}")
print(f"Median price: ${paid_games['price_dollars'].median():.2f}")
print(f"Max price: ${paid_games['price_dollars'].max():.2f}")

# Cap at $60 for the histogram — very few games cost more and they squash the chart
price_cap = 60
prices_capped = paid_games[paid_games["price_dollars"] <= price_cap]["price_dollars"]

print(f"\nShowing prices up to ${price_cap} ({len(prices_capped):,} games)")

# Build histogram
fig = go.Figure(
    go.Histogram(
        x=prices_capped,
        nbinsx=50,
        marker_color=STEAM_COLORS["accent_green"],
        hovertemplate="Price: $%{x:.2f}<br>Count: %{y:,}<extra></extra>",
    )
)

fig.update_layout(
    get_plotly_layout(
        "Price Distribution of Paid Steam Games",
        xaxis_title="Price (USD)",
        yaxis_title="Number of Games",
    )
)

fig.show()

### 4.3 Free vs Paid Games
What percentage of Steam games are free-to-play?

In [ ]:
# Count free vs paid games
free_count = games["is_free"].sum()
paid_count = len(games) - free_count

print(f"Free games: {free_count:,} ({free_count / len(games) * 100:.1f}%)")
print(f"Paid games: {paid_count:,} ({paid_count / len(games) * 100:.1f}%)")

# Build pie chart
labels = ["Paid", "Free"]
values = [paid_count, free_count]
colors = [STEAM_COLORS["light_blue"], STEAM_COLORS["accent_green"]]

fig = go.Figure(
    go.Pie(
        labels=labels,
        values=values,
        marker=dict(colors=colors, line=dict(color=STEAM_COLORS["dark_blue"], width=2)),
        textinfo="label+percent",
        textfont=dict(size=14),
        hovertemplate="%{label}: %{value:,} games<br>%{percent}<extra></extra>",
    )
)

fig.update_layout(
    get_plotly_layout("Free vs Paid Games on Steam")
)

fig.show()

### 4.4 Top 20 Genres by Game Count
Which genres are most popular among developers?

In [ ]:
# Build the games-with-genres dataset
# Each row is a game-genre pair, so a game tagged "Action, RPG" appears twice
games_with_genres = build_games_with_genres(games)

# Count how many games belong to each genre
genre_counts = (
    games_with_genres
    .groupby("genre_name")["appid"]
    .nunique()
    .sort_values(ascending=True)
    .tail(20)
)

print(f"Top genre: {genre_counts.index[-1]} with {genre_counts.values[-1]:,} games")

# Horizontal bar chart — easier to read genre names
fig = go.Figure(
    go.Bar(
        y=genre_counts.index,
        x=genre_counts.values,
        orientation="h",
        marker_color=STEAM_COLORS["light_blue"],
        hovertemplate="%{y}: %{x:,} games<extra></extra>",
    )
)

fig.update_layout(
    get_plotly_layout(
        "Top 20 Genres by Game Count",
        xaxis_title="Number of Games",
        yaxis_title="Genre",
    )
)

# Give more room for genre labels
fig.update_layout(margin=dict(l=120))

fig.show()

### 4.5 Metacritic Score Distribution
How are Metacritic scores distributed among games that have them?

In [ ]:
# Filter to games that actually have a Metacritic score
# Many games don't get reviewed by Metacritic, so this is a subset
scored_games = games[games["metacritic_score"] > 0].copy()

print(f"Games with Metacritic scores: {len(scored_games):,} out of {len(games):,}")
print(f"That's only {len(scored_games) / len(games) * 100:.1f}% of all games")
print(f"\nAverage score: {scored_games['metacritic_score'].mean():.1f}")
print(f"Median score: {scored_games['metacritic_score'].median():.1f}")
print(f"Score range: {scored_games['metacritic_score'].min()} to {scored_games['metacritic_score'].max()}")

# Build histogram
fig = go.Figure(
    go.Histogram(
        x=scored_games["metacritic_score"],
        nbinsx=40,
        marker_color=STEAM_COLORS["accent_orange"],
        hovertemplate="Score: %{x}<br>Count: %{y:,}<extra></extra>",
    )
)

fig.update_layout(
    get_plotly_layout(
        "Metacritic Score Distribution",
        xaxis_title="Metacritic Score",
        yaxis_title="Number of Games",
    )
)

fig.show()

### 4.6 Top 15 Most Recommended Games
Which games have the most community recommendations?

In [ ]:
# Find the top 15 games by total recommendations
top_recommended = (
    games[["name", "recommendations_total"]]
    .dropna(subset=["recommendations_total"])
    .sort_values("recommendations_total", ascending=True)
    .tail(15)
)

print("Top 5 most recommended games:")
for _, row in top_recommended.tail(5).iloc[::-1].iterrows():
    print(f"  {row['name']}: {int(row['recommendations_total']):,} recommendations")

# Horizontal bar chart
fig = go.Figure(
    go.Bar(
        y=top_recommended["name"],
        x=top_recommended["recommendations_total"],
        orientation="h",
        marker_color=STEAM_COLORS["accent_green"],
        hovertemplate="%{y}<br>Recommendations: %{x:,}<extra></extra>",
    )
)

fig.update_layout(
    get_plotly_layout(
        "Top 15 Most Recommended Games on Steam",
        xaxis_title="Total Recommendations",
        yaxis_title="Game",
    )
)

# Extra left margin for game titles
fig.update_layout(margin=dict(l=200))

fig.show()

## 5. Key Findings

Here's what we learned from this initial exploration:

- **The dataset is large** — tens of thousands of games with a wide range of attributes including pricing, release dates, Metacritic scores, and community recommendations.
- **Steam has grown dramatically** — the number of games released per year has skyrocketed, especially in the mid-2010s when Steam opened the floodgates with Steam Direct (replacing Steam Greenlight).
- **Most games are cheap** — the price distribution is heavily skewed toward the lower end, with a large spike around the $0.99 - $9.99 range. The median price for paid games is well below the traditional $60 AAA price point.
- **The majority of games are paid**, but free-to-play titles make up a notable minority of the catalog.
- **Indie and Action dominate** — these genres have the most games by far, reflecting the platform's accessibility to independent developers.
- **Very few games get Metacritic scores** — only a small percentage of Steam games are reviewed by professional critics, suggesting that the vast majority are smaller indie titles.
- **Community recommendations are highly concentrated** — a handful of blockbuster titles dominate the recommendation charts while most games receive very few.

### Next Steps
In the next notebook, we'll dig deeper into specific aspects like developer/publisher analysis, platform support trends, and the relationship between price, reviews, and Metacritic scores.